# 08 · Find control guides that are not inert

A control guide should have no transcriptional effect. Some do, and leaving
them in the control population biases every knockout effect measured against
it.

This notebook fits each control guide's effect on the principal components of
the control cells, then flags guides whose coefficient profile is an outlier
among control guides. Four detectors vote, and a guide flagged by more than
`par_control_guide_outlier_votes` of them is dropped.

**Reads** `par_save_filename_5`.
**Writes** `par_outlier_controlguides_recomputed_file`.

:::{note}
This notebook records how the control-guide selection was carried out.

Two of the four detectors draw random subsamples, so the exact set can differ
a little between runs. `par_control_guide_random_state` is set here so the
result is stable from one run to the next.

The list the rest of the analysis reads is
`TextFiles/OutlierControlGuides.csv`. This notebook writes its result to a
separate file and leaves that one untouched.
:::


## Setup

In [1]:
from libraries import *
from parameters import *

os.chdir(projectDir)
import statsmodels.api as sm

from sklearn.ensemble import IsolationForest
from sklearn.covariance import EllipticEnvelope
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM

## Cells carrying exactly one control guide

A cell with two control guides cannot attribute an effect to either.

In [2]:
adata = sc.read(par_save_filename_5)

control_guides = [
    g for g in adata.uns["feature_barcode_names"]
    if g.startswith((par_not_target_control_prefix, par_nongene_site_control_prefix))
]
adata.uns["Control_guides"] = control_guides
print(f"control guides: {len(control_guides)}")

carried = adata.obs[control_guides]
one_control = carried[carried.sum(axis=1) == 1]
adata_control = adata[one_control.index].copy()
print(f"cells with exactly one control guide: {adata_control.shape[0]}")

control guides: 330
cells with exactly one control guide: 44074


## Fit each guide's effect on the principal components

Regressing the components rather than the genes keeps the fit small while
retaining the dominant structure.

In [3]:
sc.tl.pca(adata_control, use_highly_variable=True,
          n_comps=par_control_guide_n_pcs, svd_solver="arpack")

expression = adata_control.obsm["X_pca"]
guide_matrix = adata_control.obs[control_guides]

coefs = pd.DataFrame()
for j in range(par_control_guide_n_pcs):
    fit = sm.OLS(np.array(expression)[:, j], np.array(guide_matrix)).fit()
    coefs[str(j)] = fit.summary2().tables[1]["Coef."]

coefs.index = control_guides
print(f"coefficient matrix: {coefs.shape[0]} guides x {coefs.shape[1]} components")

computing PCA
    on highly variable genes
    with n_comps=100
    finished (0:00:12)
coefficient matrix: 330 guides x 100 components


## Vote across four outlier detectors

No single detector is trusted on its own; a guide has to look anomalous to a
majority of them.

In [4]:
flagged = []

for name, detector in [
    ("IsolationForest", IsolationForest(contamination=par_control_guide_contamination,
                                       random_state=par_control_guide_random_state)),
    ("EllipticEnvelope", EllipticEnvelope(contamination=par_control_guide_contamination,
                                         random_state=par_control_guide_random_state)),
    ("LocalOutlierFactor", LocalOutlierFactor()),
    ("OneClassSVM", OneClassSVM(nu=0.01)),
]:
    yhat = detector.fit_predict(coefs)
    hits = list(coefs.index[yhat == -1])
    flagged.extend(hits)
    print(f"  {name:<20} flagged {len(hits)}")

votes = pd.Series(flagged).value_counts()
outliers = sorted(votes[votes > par_control_guide_outlier_votes].index)
print(f"\nflagged by more than {par_control_guide_outlier_votes} detectors: {len(outliers)}")

  IsolationForest      flagged 33


/home/eraslab1/miniconda3/lib/python3.8/site-packages/sklearn/base.py:450: UserWarning: X does not have valid feature names, but IsolationForest was fitted with feature names
  warnings.warn(


  EllipticEnvelope     flagged 33
  LocalOutlierFactor   flagged 25
  OneClassSVM          flagged 21

flagged by more than 2 detectors: 22


## Write

In [5]:
Path(par_outlier_controlguides_recomputed_file).parent.mkdir(parents=True, exist_ok=True)
pd.DataFrame({"OutlierGuides": outliers}).to_csv(par_outlier_controlguides_recomputed_file, index=False)
print(f"written: {par_outlier_controlguides_recomputed_file}")
print(f"the list used downstream is {par_outlier_controlguides_file}, left unchanged")
for g in outliers:
    print("   ", g)

written: outputs/OutlierControlGuides_recomputed.csv
the list used downstream is ./TextFiles/OutlierControlGuides.csv, left unchanged
    NO_TARGET_104
    NO_TARGET_12
    NO_TARGET_135
    NO_TARGET_149
    NO_TARGET_151
    NO_TARGET_17
    NO_TARGET_56
    NO_TARGET_64
    NO_TARGET_87
    ONE_NONGENE_SITE_174
    ONE_NONGENE_SITE_176
    ONE_NONGENE_SITE_177
    ONE_NONGENE_SITE_202
    ONE_NONGENE_SITE_207
    ONE_NONGENE_SITE_219
    ONE_NONGENE_SITE_223
    ONE_NONGENE_SITE_225
    ONE_NONGENE_SITE_263
    ONE_NONGENE_SITE_267
    ONE_NONGENE_SITE_279
    ONE_NONGENE_SITE_300
    ONE_NONGENE_SITE_302
